# NCP-Wired CfC Phase 1 - Google Colab

Bu notebook, NCP-wired CfC mimarisini Colab'da sifirdan kurar, BC + DAgger ile egitir ve 22 custom + 2 dynamic haritada degerlendirir.

Kullanim: `Runtime > Change runtime type > GPU` sec, sonra hucreleri ustten alta calistir. `RESUME_CHECKPOINT` yok; bu akis sifirdan egitim icindir.

## 1. GPU kontrolu

In [ ]:
!nvidia-smi

## 2. Repoyu klonla ve kur

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/heimdilon/mujoco_lnn_navigation.git"
REPO_DIR = Path("/content/mujoco_lnn_navigation")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/mujoco_lnn_navigation
!git checkout master
!git pull --ff-only

In [ ]:
%env MUJOCO_GL=egl
!apt-get update -qq
!apt-get install -y -qq libegl1 libgl1
!python -m pip install -U pip
!python -m pip install -e .
!python -m pip install matplotlib

## 3. Ortami dogrula

In [ ]:
import importlib.metadata as metadata
import torch

print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
print("cuda_device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ncps", metadata.version("ncps"))

In [ ]:
!python tools/ncp_smoke_test.py

## 4. NCP Phase 1 egitimi

In [ ]:
import torch

RUN_NAME = "ncp_cfc_phase1"
SPLIT_CONFIG = "configs/splits/custom22_dynamic_seed25462877008.yaml"
TRAIN_CONFIG = "configs/train/bc_ncp_cfc_dynamic_maps.yaml"
DAGGER_ITERATIONS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("run", RUN_NAME)
print("device", DEVICE)

In [ ]:
import shlex
import subprocess
import sys

train_cmd = [
    sys.executable,
    "scripts/train_bc.py",
    "--split-config",
    SPLIT_CONFIG,
    "--train-config",
    TRAIN_CONFIG,
    "--run-name",
    RUN_NAME,
    "--dagger-iterations",
    str(DAGGER_ITERATIONS),
    "--save-interval",
    "50",
    "--no-final-eval",
    "--device",
    DEVICE,
]
print(" ".join(shlex.quote(part) for part in train_cmd))
subprocess.run(train_cmd, check=True)

## 5. 24 harita degerlendirme

In [ ]:
import yaml

EVAL_RUN_NAME = f"{RUN_NAME}_eval_all24"
EVAL_EPISODES = 4
MAKE_VISUALS = False

with open(SPLIT_CONFIG, "r", encoding="utf-8") as handle:
    split = yaml.safe_load(handle)

map_paths = list(split["train_maps"]) + list(split["holdout_maps"])
print(len(map_paths), "maps")

In [ ]:
eval_cmd = [
    sys.executable,
    "scripts/batch_evaluate.py",
    "--map-configs",
    *map_paths,
    "--checkpoint",
    f"results/{RUN_NAME}/latest.pt",
    "--run-name",
    EVAL_RUN_NAME,
    "--episodes",
    str(EVAL_EPISODES),
    "--max-steps",
    "900",
    "--goal-observation-max",
    "10.0",
    "--device",
    "cpu",
]
if not MAKE_VISUALS:
    eval_cmd.extend(["--no-gif", "--no-png"])

print(" ".join(shlex.quote(part) for part in eval_cmd))
subprocess.run(eval_cmd, check=True)

In [ ]:
import pandas as pd

summary_path = Path("results") / EVAL_RUN_NAME / "summary.csv"
summary = pd.read_csv(summary_path)
display(summary[["map", "success_rate", "collision_rate", "timeout_rate", "mean_steps"]])

success_maps = int((summary["success_rate"] > 0.0).sum())
print(f"success maps: {success_maps}/{len(summary)}")
print("summary:", summary_path)

## 6. Sonuclari indir

In [ ]:
from google.colab import files

zip_name = f"{RUN_NAME}_artifacts.zip"
!zip -r -q {zip_name} results/{RUN_NAME} results/{EVAL_RUN_NAME}
files.download(zip_name)